# Chapter 1: spaCy — Industrial-Strength Classic NLP

### Why This Matters

Before reaching for a large, expensive language model, a huge amount of practical business text processing — extracting company names from a report, pulling dates and amounts from contracts, routing support tickets by keyword — is solved faster, cheaper, and more reliably by spaCy's classic NLP techniques. spaCy is optimized for production speed on large text volumes, not research flexibility.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")     # a small pretrained English pipeline (installed via: python -m spacy download en_core_web_sm)

text = """Apple is planning to open a new store in Mumbai in 2026.
CEO Tim Cook announced an investment of $50 million."""

doc = nlp(text)

print("Named Entities:")
for ent in doc.ents:
    print(f"  {ent.text:20} -> {ent.label_}")

print("\nNoun chunks:", [chunk.text for chunk in doc.noun_chunks])

Calling `nlp(text)` runs the entire linguistic pipeline in one step (tokenization, part-of-speech tagging, dependency parsing, named entity recognition) and returns a `Doc` object you can query in multiple ways. **Named Entity Recognition (NER)** — `doc.ents` — automatically identifies and categorizes real-world entities (people, organizations, locations, money amounts, dates) in raw text, exactly the "extract structured facts from unstructured text" capability that underlies invoice processing, contract analysis, and news monitoring at a business scale. **Noun chunks** identify the core noun phrases in a sentence — useful for a fast, cheap first pass at "what is this text actually about" before committing to heavier analysis.

### 1.1 Tokenization — The Foundational Step

In [ ]:
nlp_blank = spacy.blank("en")     # a bare tokenizer, no trained model needed — fast, no download required
doc = nlp_blank("OpenAI launched a new model in San Francisco.")
print([(t.text, t.pos_) for t in doc])

`spacy.blank("en")` gives you tokenization rules for English without loading a full trained statistical model — useful when you only need to split text into words/punctuation quickly, without the overhead (and download size) of the full pipeline. **Tokenization** — splitting raw text into discrete units (words, punctuation, sometimes sub-words) — is the essential first step of virtually every NLP task in this entire volume, including the transformer-based methods in Chapter 2.

### 1.2 Rule-Based Matching — When You Know Exactly What You're Looking For

In [ ]:
from spacy.matcher import PhraseMatcher

matcher = PhraseMatcher(nlp_blank.vocab)
patterns = [nlp_blank.make_doc("San Francisco"), nlp_blank.make_doc("OpenAI")]
matcher.add("ORG_OR_PLACE", patterns)

doc = nlp_blank("OpenAI launched an office in San Francisco.")
print([(doc[start:end].text, match_id) for match_id, start, end in matcher(doc)])

Statistical NER (Section 1's `doc.ents`) is powerful but probabilistic — it can miss or misclassify entities, especially unusual company names or domain-specific terms it wasn't trained on. **Rule-based matching** is the deliberate, precise complement: when you know exactly which specific terms matter (your own product names, a fixed list of competitor companies, specific policy phrases), a `PhraseMatcher` finds them with 100% precision on that exact list — no statistical uncertainty at all. Real production NLP pipelines routinely combine both: statistical NER for open-ended discovery, rule-based matching for known, business-critical terms that must never be missed.

### Cheat Sheet — spaCy

| Task | Code |
|---|---|
| Load a pipeline | `spacy.load("en_core_web_sm")` |
| Process text | `doc = nlp(text)` |
| Named entities | `doc.ents`, each with `.text` and `.label_` |
| Noun phrases | `doc.noun_chunks` |
| Bare tokenizer (no model) | `spacy.blank("en")` |
| Exact phrase matching | `PhraseMatcher(nlp.vocab)` |

---